# bgm_stable_audio

**Stable Audio 3 Medium** 으로 BGM 생성.

| | |
|---|---|
| 커널 | `sa3` |
| 출력 | `outputs/bgm_stable_audio/<시나리오>/` |
| 개수 | 시나리오당 페이즈 2 x 후보 3 = 6개 |

## 페이즈가 참조하는 이야기가 다르다

- `story` ← `source_story` (원작 설화, 사건 전)
- `case`  ← `adapted_story` (각색 사건, 사건 후)

둘을 같은 톤으로 만들면 사건 전부터 살인 분위기가 깔려 낙차가 사라진다.

## 사전 준비 (1회)

1. huggingface.co/stabilityai/stable-audio-3-medium 에서 **약관 동의**
2. Settings → Access Tokens 에서 read 토큰 발급
3. 아래 2번 셀에 토큰 입력

gated repo라 동의·토큰 없이는 다운로드 단계에서 401이 난다.


## 1. 커널 확인

설치는 `setup_kernels.ipynb` 가 한다. 여기서는 커널이 맞는지만 본다.
커널을 `sa3` 로 바꾸고 실행할 것.

In [ ]:
# ---------------------------------------------------------------------------
# 커널 확인. 설치는 setup_kernels.ipynb 가 담당한다.
#
# 이 셀에 pip install 을 추가하지 말 것. 노트북 4개는 서로 양립 불가능한
# torch/transformers 를 쓰기 때문에, 여기서 다시 깔면 커널 조합이 깨진다.
#   ace  torch 2.10.0+cu128 / transformers >=4.51,<4.58
#   sa3  torch 2.7.1 cu126  / transformers >=5.8
#   flux torch 2.8.0        / transformers ==4.56.1
# ---------------------------------------------------------------------------
import sys, importlib

KERNEL = "sa3"
NEED   = ["torch", "torchaudio", "soundfile", "stable_audio_3"]

print("python     :", sys.version.split()[0])
print("interpreter:", sys.executable)
print()

missing = []
for m in NEED:
    try:
        mod = importlib.import_module(m)
        print(f"OK   {m:16s} {getattr(mod, '__version__', '')}")
    except Exception as e:
        missing.append(m)
        print(f"FAIL {m:16s} {type(e).__name__}: {e}")

assert not missing, (
    f"\n{missing} 를 import 하지 못했다.\n"
    f"커널을 '{KERNEL}' 로 바꿀 것 (Kernel > Change kernel).\n"
    f"'{KERNEL}' 커널이 목록에 없으면 setup_kernels.ipynb 를 먼저 실행한다.\n"
    f"여기서 pip install 로 때우지 말 것 — 버전 조합이 깨진다."
)
print("\n커널 OK")

## 2. 인증

Hugging Face 토큰. 저장소에 커밋하지 말 것.


In [ ]:
import os, pathlib

# 토큰을 노트북에 적지 않는다. 커밋되면 그대로 노출된다.
# 터미널에서:  export HF_TOKEN=hf_xxx   (커널을 띄우기 전에. 이미 떠 있으면 커널 재시작)
# 또는 파일로:  echo hf_xxx > ~/.hf_token
tok = os.environ.get("HF_TOKEN", "").strip()
if not tok:
    p = pathlib.Path.home() / ".hf_token"
    if p.exists():
        tok = p.read_text().strip()
        os.environ["HF_TOKEN"] = tok

assert tok.startswith("hf_"), (
    "HF_TOKEN 이 없다.\n"
    "  export HF_TOKEN=hf_...   후 커널 재시작\n"
    "  또는  echo hf_... > ~/.hf_token\n"
    "gated repo 라 토큰 없이는 401 이 난다."
)
print("토큰 설정됨:", tok[:6] + "…" + f" ({len(tok)}자)")

## 3. 설정


In [ ]:
import os, sys, json, pathlib, getpass, shutil

USER = getpass.getuser()

# ---- 이 노드에서 유효한 루트를 직접 찾는다 --------------------------------
# 학과 클러스터는 같은 저장소가 노드마다 다른 경로에 마운트된다.
#   로그인 노드(hpcmaster) : /data1/$USER   (= HOME)
#   계산 노드(hpc-stat1)   : /mnt/data1/$USER
# setup_kernels 로 만든 커널이면 PROJ_BASE 가 kernel.json 에 박혀 있고,
# Slurm 으로 돌리면 common.sh 가 환경변수로 넘겨준다. 둘 다 아니면 여기서 찾는다.
def pick_root():
    for c in (f"/mnt/data1/{USER}", f"/data1/{USER}",
              str(pathlib.Path.home()), "/workspace", "/scratch"):
        if pathlib.Path(c).is_dir() and os.access(c, os.W_OK):
            return c
    return None

ROOT = os.environ.get("PROJ_ROOT") or pick_root()
if ROOT is None:
    raise SystemExit(f"쓸 수 있는 루트를 못 찾았다 (user={USER})")

# 계산 노드에는 $HOME 이 아예 없다. 두면 캐시/kernelspec 이 전부 깨진다.
if not pathlib.Path(os.environ.get("HOME", "/nonexistent")).is_dir():
    os.environ["HOME"] = ROOT

BASE  = pathlib.Path(os.environ.get("PROJ_BASE") or f"{ROOT}/mystery")
MODEL = "bgm_stable_audio"
OUT_DIR = BASE / "outputs" / MODEL

# 캐시는 홈이 아니라 대용량 스토리지로. 홈 쿼터면 다운로드 중 os error 28 이 난다.
os.environ.setdefault("HF_HOME",      str(BASE / "hf_cache"))
os.environ.setdefault("TMPDIR",       str(BASE / "tmp"))
os.environ.setdefault("MPLCONFIGDIR", str(BASE / "mpl_cache"))

try:
    for p in (OUT_DIR, pathlib.Path(os.environ["HF_HOME"]),
              pathlib.Path(os.environ["TMPDIR"])):
        p.mkdir(parents=True, exist_ok=True)
except (PermissionError, OSError) as e:
    raise SystemExit(
        f"{BASE} 에 쓸 수 없다 -> {type(e).__name__}: {e}\n"
        f"  이 셀 맨 위에서 지정할 것:\n"
        f"      os.environ['PROJ_BASE'] = '/data1/{USER}/mystery'"
    )

# ---- 입력 JSON (scenarios_en/*.json 전부) ---------------------------------
# papermill 로 돌리면 cwd 가 /tmp 라 "." 만으로는 못 찾는다.
# PROJ_INPUT(환경변수) 을 먼저 보고, 없으면 후보들을 뒤진다.
def pick_input_dir():
    cands = []
    if os.environ.get("PROJ_INPUT"):
        cands.append(pathlib.Path(os.environ["PROJ_INPUT"]))
    cands += [pathlib.Path.cwd(), pathlib.Path.cwd().parent, BASE]
    for c in cands:
        if (c / "scenarios_en").is_dir() and list((c / "scenarios_en").glob("*.json")):
            return c
    # 마지막 수단: ROOT 아래를 얕게 훑는다
    for c in pathlib.Path(ROOT).glob("*/*/"):
        if (c / "scenarios_en").is_dir() and list((c / "scenarios_en").glob("*.json")):
            return c
    return None

IN_DIR = pick_input_dir()
if IN_DIR is None:
    raise SystemExit(
        "scenarios_en/*.json 을 못 찾았다.\n"
        f"  PROJ_INPUT = {os.environ.get('PROJ_INPUT', '(미설정)')}\n"
        f"  cwd        = {pathlib.Path.cwd()}\n"
        f"  이 셀 맨 위에서:  os.environ['PROJ_INPUT'] = '/data1/{USER}/<저장소>/Jaemin'"
    )

# scenario_id(파일 내부 필드, 예: "12_heungbu")를 키로 쓴다 — 파일명과 무관하게 안정적.
SCENARIO_FILES = sorted((IN_DIR / "scenarios_en").glob("*.json"))
assert SCENARIO_FILES, f"scenarios_en 에 json 이 없다: {IN_DIR / 'scenarios_en'}"

DATA = {}
for _f in SCENARIO_FILES:
    _d = json.load(open(_f, encoding="utf-8"))
    DATA[_d["scenario_id"]] = _d

for k in DATA:
    (OUT_DIR / k).mkdir(parents=True, exist_ok=True)


def out_path(scen, *parts):
    """OUT_DIR/<시나리오>/<...> 경로를 만들고 돌려준다."""
    p = OUT_DIR / scen
    for x in parts[:-1]:
        p = p / x
    p.mkdir(parents=True, exist_ok=True)
    return p / parts[-1]


print("node    :", os.uname().nodename)
print("ROOT    :", ROOT)
print("BASE    :", BASE, f"(여유 {shutil.disk_usage(BASE).free/1e9:.0f} GB)")
print("IN_DIR  :", IN_DIR)
print("HF_HOME :", os.environ["HF_HOME"])
print("OUT_DIR :", OUT_DIR)
print(f"시나리오 {len(DATA)}개:")
for k, d in DATA.items():
    print(f"  {k:24s} 장소 {len(d['places'])} / 인물 {len(d['cast'])}")


In [ ]:
import gc, torch

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
DTYPE  = torch.bfloat16 if DEVICE == "cuda" else torch.float32

print("torch  :", torch.__version__)
print("device :", DEVICE)
if DEVICE == "cuda":
    p = torch.cuda.get_device_properties(0)
    print("GPU    :", p.name, "|", round(p.total_memory / 1e9, 1), "GB",
          "| compute capability", f"{p.major}.{p.minor}")
    if (p.major, p.minor) < (8, 9):
        print("  주의: FP8 연산 미지원 세대. FP8 가중치를 요구하는 모델은 못 돌린다.")
else:
    print("  GPU가 없다. 로그인 노드이거나 GPU 할당이 안 된 상태.")


def free():
    """모델을 내린 뒤 호출. 커널을 종료하는 것이 가장 확실하다."""
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache(); torch.cuda.ipc_collect()
    if torch.cuda.is_available():
        used = torch.cuda.memory_allocated() / 1e9
        print(f"할당된 VRAM: {used:.2f} GB")


## 4. 프롬프트 조립

입력 JSON이 이미 영어라 템플릿에 끼우기만 한다.


In [ ]:
PHASES = [
    {"id": "story", "energy": "low", "source": "source_story"},
    {"id": "case", "energy": "mid", "source": "adapted_story"},
]

# 조건만으로 문화가 안 잡히면 여기에 악기를 직접 넣는다.
INSTRUMENT_OVERRIDE = {
    # "heungbu": "daegeum bamboo flute, haegeum fiddle, geomungo zither, janggu drum, "
    #            "jing gong, traditional Korean instrumentation, gyemyeonjo mode",
}


# 실측 확인(2026-08-24): stable-audio-3 "medium" 은 post-trained 체크포인트라
# 공식 문서(docs/workflows/inference.md)는 steps/cfg_scale 이 base 모델에만
# 의미가 있다고 안내하지만, 실제로 cfg_scale=1 vs 7 로 서로 다른 출력이 나오고
# (md5 다름), cfg_scale=7 에서는 negative_prompt 유무로도 출력이 달라진다
# (cfg_scale=1 에서는 negative_prompt 가 진짜로 무효 — CFG 공식상 당연한 결과).
# 즉 cfg_scale=7 + negative_prompt 조합이 실제로 작동하는 손잡이다.
BASE_NEGATIVE = ("distorted, harsh dissonance, atonal noise, static, glitch, "
                  "poor audio quality, muddy recording, out of tune, screeching feedback")
CALM_NEGATIVE_EXTRA = ", scary, horror, ominous dread, frightening, unsettling"


def bgm_conditions(scen, d):
    era, loc = d["setting"]["era"], d["setting"]["location"]
    out = {}
    for p in PHASES:
        src = d[p["source"]]
        tone = src["tone"]
        # 악기 지정을 맨 앞쪽에 둔다 — stable-audio-3 의 T5Gemma 텍스트 인코더는
        # max_length=256 토큰에서 truncation=True 로 뒷부분을 그대로 잘라버린다.
        # synopsis(줄거리 서술)를 프롬프트에 섞으면 200토큰을 넘기기 쉬워서,
        # 정작 중요한 "instrumentation: ..." 부분이 통째로 잘려나가는 사고가
        # 실제로 있었다(예: 흥부전 story 트랙에서 janggu drum/gyemyeonjo 부분이
        # 잘림 -> "악기 구성이 프롬프트와 안 맞는다"는 증상으로 나타남).
        # ace_step 은 애초에 synopsis 를 안 쓰는 태그형 프롬프트라 이 문제가 없다
        # (bgm_ace_step.ipynb 의 ace_tags() 참고) — 여기서도 같은 방식을 따른다.
        instr_str = d.get("culture", {}).get("instruments") or INSTRUMENT_OVERRIDE.get(scen)
        prompt = "instrumental background music, no vocals, seamless loop, sparse arrangement."
        if instr_str:
            prompt += f" instrumentation: {instr_str}."
        prompt += (f" mood: {tone}. setting: {era}, {loc}. "
                   f"energy: {p['energy']}.")
        # story(원작)는 밝고 향수 어린 톤이 많다 — "무섭게" 나오면 안 되니
        # scary/horror 계열도 배제한다. case(각색 사건)는 애초에 긴장·불길함이
        # 의도된 톤이라 그것까지 막으면 오히려 밋밋해진다.
        negative = BASE_NEGATIVE + (CALM_NEGATIVE_EXTRA if p["id"] == "story" else "")
        out[p["id"]] = {"source": p["source"], "tone": tone, "prompt": prompt, "negative_prompt": negative}
    return out


COND = {k: bgm_conditions(k, d) for k, d in DATA.items()}

N       = 3        # 페이즈당 후보
SECONDS = 60
SR      = 44100
TAG     = ""       # 조건 바꿔 재생성할 때 "_B" 등으로. 이전 결과를 안 덮는다.

for k, ph in COND.items():
    print("=" * 70); print(k)
    for pid, v in ph.items():
        print(f"  [{pid}] <- {v['source']} | {v['tone']}")
print()
print(f"시나리오 {len(COND)} x 페이즈 {len(PHASES)} x 후보 {N} = "
      f"{len(COND) * len(PHASES) * N}개 생성 예정")


## 5. 생성


In [ ]:
from stable_audio_3 import StableAudioModel
import soundfile as sf
import numpy as np
import torch

sa3 = StableAudioModel.from_pretrained("medium", device=DEVICE)

# 저장 샘플레이트는 모델이 알려주면 그걸 따른다. 헤더에 틀린 값을 쓰면
# 에러 없이 재생 속도와 음높이가 어긋난다.
SR_OUT = int(getattr(sa3, "sample_rate", SR))
print("저장 샘플레이트:", SR_OUT, "(모델이 보고한 값)" if hasattr(sa3, "sample_rate") else "(모델이 안 알려줘 기본값 사용)")

for scen, ph in COND.items():
    for pid, v in ph.items():
        for i in range(N):
            audio = sa3.generate(prompt=v["prompt"], negative_prompt=v["negative_prompt"],
                                 duration=SECONDS, steps=8, cfg_scale=3.0, seed=i)
            # (batch, channels, samples) -> (samples, channels)
            wav = audio[0]
            if isinstance(wav, torch.Tensor):
                wav = wav.T.float().cpu().numpy()
            else:
                wav = np.asarray(wav).T
            # [패치] 모델 원본 출력을 정규화 없이 그대로 저장하면 진폭이 ±1 을
            # 넘는 구간에서 하드 클리핑이 나 지지직거리는 소리가 난다(실측:
            # 60초 파일 하나에 최대 22만 샘플, 전체의 최대 8% 클리핑).
            # ACE-Step 은 자체 파이프라인에서 -1dB 로 피크 정규화하는데 여기는
            # 그게 없었다. 같은 기준(-1dBFS)으로 맞춰서 클리핑을 막는다.
            peak = np.abs(wav).max()
            if peak > 0:
                target_peak = 10 ** (-1.0 / 20)  # -1 dBFS, ACE-Step과 동일 기준
                wav = wav / peak * target_peak
            wav = np.clip(wav, -1.0, 1.0)  # 안전망
            sf.write(out_path(scen, f"{pid}{TAG}_{i}.wav"), wav, SR_OUT)
        print("OK", scen, pid, f"({N}개)")

print("\n저장 위치:", OUT_DIR)

In [ ]:
del sa3
free()


## 6. 청취 체크

1. 시나리오 분위기에 맞는가
2. **시대·문화가 반영되는가** — 흥부놀부에서 국악기가 들리는가
3. **story와 case가 충분히 다른가** — 원작 톤과 사건 톤의 낙차
4. **보컬 누출** — 심문 중 사람 목소리가 섞이면 대사와 충돌한다
5. **루프 이음새** — `case`는 수십 분 반복된다


In [ ]:
from IPython.display import Audio, display
import glob

def listen(scen, pattern="*"):
    fs = sorted(glob.glob(str(OUT_DIR / scen / f"{pattern}.wav")))
    if not fs:
        print("없음:", scen, pattern); return
    for f in fs:
        print(f.split("/")[-1]); display(Audio(f))

demo_scen = next(iter(DATA))
print(f"### {demo_scen} · 원작(story)")
listen(demo_scen, "story*")
print(f"### {demo_scen} · 사건 후(case)")
listen(demo_scen, "case*")
